# 0. Summary
- Download complete HUBEAU_DATA_ENDPOINT data from stations of site SITE.
- Convert "date_obs_elab" to datetime
- Drop "date_prod" column
- Drop raw data (code_statut == 4)
- Drop non-qualified data (code_qualification != 20)
- Drop "code_statut" and "code_statut" columns
- Drop "code_methode" and "libelle_methode" columns
- Drop "code_qualification" and "libelle_qualification" columns
- Identify min and max dates
- Save to LOCAL_DATA_DIRECTORY as "hubeau_{HUBEAU_DATA_ENDPOINT}_{SITE}_{min_date}_{max_date}_1.csv" 

In [ ]:
import os
import pandas
import datetime

# LOCAL COMPONENTS:
from config import HUBEAU_BASE_URL, SITES
from utils import request_json_all, save_dataframe_as_csv

# 1. Download data

In [ ]:
HUBEAU_DATA_ENDPOINT = "obs_elab"

In [127]:
SITE = SITES["Selestat"]["code"]
STATIONS = ",".join([station["code"] for station in SITES["Selestat"]["stations"]])

In [128]:
print("Site:", SITE, "/", "Stations:", STATIONS)

Site: A2350200 / Stations: A235020001,A235020002,A235020003


In [129]:
url = f"{HUBEAU_BASE_URL}/{HUBEAU_DATA_ENDPOINT}"
params = {"code_entite": STATIONS, "size": 2000}  
results = request_json_all(url, params=params, page_timeout_in_seconds=300)

In [130]:
print(type(results), results.keys())
assert(all(key in results.keys() for key in ("api_version", "count", "data")))
data = results["data"]
print(type(data), len(data))
assert(isinstance(data, list))
assert(len(data) > 0)
data_0 = data[0]
print(type(data_0), data_0.keys())

<class 'dict'> dict_keys(['api_version', 'count', 'data'])
<class 'list'> 116844
<class 'dict'> dict_keys(['code_site', 'code_station', 'date_obs_elab', 'resultat_obs_elab', 'date_prod', 'code_statut', 'libelle_statut', 'code_methode', 'libelle_methode', 'code_qualification', 'libelle_qualification', 'longitude', 'latitude', 'grandeur_hydro_elab'])


In [131]:
df = pandas.DataFrame(data)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 116844 entries, 0 to 116843
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   code_site              116844 non-null  str    
 1   code_station           116844 non-null  str    
 2   date_obs_elab          116844 non-null  str    
 3   resultat_obs_elab      116844 non-null  float64
 4   date_prod              116844 non-null  str    
 5   code_statut            116844 non-null  int64  
 6   libelle_statut         116844 non-null  str    
 7   code_methode           116844 non-null  int64  
 8   libelle_methode        116844 non-null  str    
 9   code_qualification     116844 non-null  int64  
 10  libelle_qualification  116844 non-null  str    
 11  longitude              116844 non-null  float64
 12  latitude               116844 non-null  float64
 13  grandeur_hydro_elab    116844 non-null  str    
dtypes: float64(3), int64(3), str(8)
memory usage: 2

In [132]:
display(df.head())

,code_site,code_station,date_obs_elab,resultat_obs_elab,date_prod,code_statut,libelle_statut,code_methode,libelle_methode,code_qualification,libelle_qualification,longitude,latitude,grandeur_hydro_elab
0,A2350200,A235020001,2007-01-01,958.0,2026-05-31T21:42:40Z,16,Donnée validée,10,Expertisée,20,Bonne,7.459819,48.271833,HIXM
1,A2350200,A235020001,2007-02-01,1079.0,2026-05-31T21:42:40Z,16,Donnée validée,10,Expertisée,20,Bonne,7.459819,48.271833,HIXM
2,A2350200,A235020001,2007-03-01,1476.0,2026-05-31T21:42:40Z,16,Donnée validée,10,Expertisée,20,Bonne,7.459819,48.271833,HIXM
3,A2350200,A235020001,2007-04-01,754.0,2026-05-31T21:42:40Z,16,Donnée validée,10,Expertisée,20,Bonne,7.459819,48.271833,HIXM
4,A2350200,A235020001,2007-05-01,783.0,2026-05-31T21:42:40Z,16,Donnée validée,10,Expertisée,20,Bonne,7.459819,48.271833,HIXM


# 2. Convert dates to datetime

In [ ]:
## ignore data-production's dates:
df.drop(columns=["date_prod"], inplace=True)  

In [ ]:
## convert observations' dates to datetime:
df["date_obs_elab"] = pandas.to_datetime(df["date_obs_elab"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 116844 entries, 0 to 116843
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   code_site              116844 non-null  str           
 1   code_station           116844 non-null  str           
 2   date_obs_elab          116844 non-null  datetime64[us]
 3   resultat_obs_elab      116844 non-null  float64       
 4   code_statut            116844 non-null  int64         
 5   libelle_statut         116844 non-null  str           
 6   code_methode           116844 non-null  int64         
 7   libelle_methode        116844 non-null  str           
 8   code_qualification     116844 non-null  int64         
 9   libelle_qualification  116844 non-null  str           
 10  longitude              116844 non-null  float64       
 11  latitude               116844 non-null  float64       
 12  grandeur_hydro_elab    116844 non-null  str           


In [135]:
display(df.head())

,code_site,code_station,date_obs_elab,resultat_obs_elab,code_statut,libelle_statut,code_methode,libelle_methode,code_qualification,libelle_qualification,longitude,latitude,grandeur_hydro_elab
0,A2350200,A235020001,2007-01-01,958.0,16,Donnée validée,10,Expertisée,20,Bonne,7.459819,48.271833,HIXM
1,A2350200,A235020001,2007-02-01,1079.0,16,Donnée validée,10,Expertisée,20,Bonne,7.459819,48.271833,HIXM
2,A2350200,A235020001,2007-03-01,1476.0,16,Donnée validée,10,Expertisée,20,Bonne,7.459819,48.271833,HIXM
3,A2350200,A235020001,2007-04-01,754.0,16,Donnée validée,10,Expertisée,20,Bonne,7.459819,48.271833,HIXM
4,A2350200,A235020001,2007-05-01,783.0,16,Donnée validée,10,Expertisée,20,Bonne,7.459819,48.271833,HIXM


# 3. Check maximal observation dates (must be greater or equal than 2025-12-31)

In [136]:
## 2.1. Check that we have the three stations:
df["code_station"].value_counts()

code_station
A235020002    61544
A235020001    29304
A235020003    25996
Name: count, dtype: int64

In [137]:
# max date = 2026/06 -> ok
df_01 = df[df["code_station"] == "A235020001"]
display(df_01.describe())

,date_obs_elab,resultat_obs_elab,code_statut,code_methode,code_qualification,longitude,latitude
count,29304,29304.000000,29304.000000,29304.000000,29304.000000,2.930400e+04,2.930400e+04
mean,2016-09-15 11:55:28.746928,2246.017199,15.912367,9.795386,19.525799,7.459819e+00,4.827183e+01
min,2007-01-01 00:00:00,-83.000000,4.000000,0.000000,12.000000,7.459819e+00,4.827183e+01
25%,2011-11-08 00:00:00,464.000000,16.000000,10.000000,20.000000,7.459819e+00,4.827183e+01
50%,2016-09-15 12:00:00,809.000000,16.000000,10.000000,20.000000,7.459819e+00,4.827183e+01
75%,2021-07-25 00:00:00,2516.000000,16.000000,10.000000,20.000000,7.459819e+00,4.827183e+01
max,2026-06-02 00:00:00,67939.000000,16.000000,10.000000,20.000000,7.459819e+00,4.827183e+01
std,NaN,3951.128632,0.602097,1.118894,1.888277,3.552774e-15,1.421110e-14


In [138]:
# max date = 2007 -> can be ignored
df_02 = df[df["code_station"] == "A235020002"]
display(df_02.describe())

,date_obs_elab,resultat_obs_elab,code_statut,code_methode,code_qualification,longitude,latitude
count,61544,61544.000000,61544.000000,61544.000000,61544.000000,6.154400e+04,6.154400e+04
mean,1986-07-05 20:48:12.447679,2866.943699,15.900819,9.998083,19.990641,7.456647e+00,4.827284e+01
min,1965-12-21 00:00:00,0.000000,12.000000,4.000000,12.000000,7.456647e+00,4.827284e+01
25%,1976-03-19 00:00:00,509.000000,16.000000,10.000000,20.000000,7.456647e+00,4.827284e+01
50%,1986-08-12 12:00:00,1016.500000,16.000000,10.000000,20.000000,7.456647e+00,4.827284e+01
75%,1996-10-23 00:00:00,2943.000000,16.000000,10.000000,20.000000,7.456647e+00,4.827284e+01
max,2007-01-03 00:00:00,153000.000000,16.000000,10.000000,20.000000,7.456647e+00,4.827284e+01
std,NaN,5635.517098,0.622008,0.106021,0.273472,1.776371e-15,2.131646e-14


In [139]:
df = df[df["code_station"] != "A235020002"]

In [140]:
# max date = 1988 -> can be ignored
df_03 = df[df["code_station"] == "A235020003"]
display(df_03.describe())

,date_obs_elab,resultat_obs_elab,code_statut,code_methode,code_qualification,longitude,latitude
count,25996,25996.000000,25996.0,25996.000000,25996.000000,25996.000000,25996.000000
mean,1979-10-09 22:06:00.055393,2587.751423,16.0,9.999231,19.994768,7.475315,48.278697
min,1971-01-01 00:00:00,0.000000,16.0,8.000000,12.000000,7.475315,48.278697
25%,1975-06-13 00:00:00,306.000000,16.0,10.000000,20.000000,7.475315,48.278697
50%,1979-10-12 12:00:00,860.000000,16.0,10.000000,20.000000,7.475315,48.278697
75%,1984-02-04 00:00:00,2862.000000,16.0,10.000000,20.000000,7.475315,48.278697
max,1988-12-31 00:00:00,97000.000000,16.0,10.000000,20.000000,7.475315,48.278697
std,NaN,5021.296993,0.0,0.039219,0.188870,0.000000,0.000000


In [141]:
df = df[df["code_station"] != "A235020003"]

In [142]:
display(df.describe())

,date_obs_elab,resultat_obs_elab,code_statut,code_methode,code_qualification,longitude,latitude
count,29304,29304.000000,29304.000000,29304.000000,29304.000000,2.930400e+04,2.930400e+04
mean,2016-09-15 11:55:28.746928,2246.017199,15.912367,9.795386,19.525799,7.459819e+00,4.827183e+01
min,2007-01-01 00:00:00,-83.000000,4.000000,0.000000,12.000000,7.459819e+00,4.827183e+01
25%,2011-11-08 00:00:00,464.000000,16.000000,10.000000,20.000000,7.459819e+00,4.827183e+01
50%,2016-09-15 12:00:00,809.000000,16.000000,10.000000,20.000000,7.459819e+00,4.827183e+01
75%,2021-07-25 00:00:00,2516.000000,16.000000,10.000000,20.000000,7.459819e+00,4.827183e+01
max,2026-06-02 00:00:00,67939.000000,16.000000,10.000000,20.000000,7.459819e+00,4.827183e+01
std,NaN,3951.128632,0.602097,1.118894,1.888277,3.552774e-15,1.421110e-14


# 4. Explore and prune other columns

In [ ]:
## Explore "qualification"
display(df["libelle_qualification"].value_counts())
display(df["code_qualification"].value_counts())

libelle_qualification
Bonne            27564
Douteuse          1734
Non qualifiée        6
Name: count, dtype: int64

code_qualification
20    27564
12     1734
16        6
Name: count, dtype: int64

In [144]:
## 3.1. Explore "status"
display(df["libelle_statut"].value_counts())
display(df["code_statut"].value_counts())

libelle_statut
Donnée validée        28674
Donnée pré-validée      624
Donnée brute              6
Name: count, dtype: int64

code_statut
16    28674
12      624
4         6
Name: count, dtype: int64

In [145]:
## 3.3. Explore "method"
display(df["libelle_methode"].value_counts())
display(df["code_methode"].value_counts())

libelle_methode
Expertisée      28002
Calculée          628
Reconstituée      500
Mesurée           174
Name: count, dtype: int64

code_methode
10    28002
8       628
4       500
0       174
Name: count, dtype: int64

In [ ]:
## 3.4. Ignore non-qualified data
df = df[df["code_qualification"] == 20]
display(df["code_methode"].value_counts())
display(df["code_statut"].value_counts())

code_qualification
20    27564
Name: count, dtype: int64

code_statut
16    26940
12      624
Name: count, dtype: int64

In [147]:
## 3.5. Check that current records are not raw (validated or pre-validated)
assert((df["code_statut"] == 4).sum() == 0)

In [148]:
## 3.6. Check that current records are all qualified
display(df["libelle_statut"].value_counts())
display(df["libelle_methode"].value_counts())
display(df["libelle_qualification"].value_counts())

libelle_statut
Donnée validée        26940
Donnée pré-validée      624
Name: count, dtype: int64

libelle_methode
Expertisée      26622
Reconstituée      500
Calculée          323
Mesurée           119
Name: count, dtype: int64

libelle_qualification
Bonne    27564
Name: count, dtype: int64

In [149]:
## 3.7. Remove unused columns
df.drop(columns=["libelle_statut", "code_statut", "libelle_methode", "code_methode", "libelle_qualification", "code_qualification"], inplace=True)
display(df)

,code_site,code_station,date_obs_elab,resultat_obs_elab,longitude,latitude,grandeur_hydro_elab
0,A2350200,A235020001,2007-01-01,958.0,7.459819,48.271833,HIXM
1,A2350200,A235020001,2007-02-01,1079.0,7.459819,48.271833,HIXM
2,A2350200,A235020001,2007-03-01,1476.0,7.459819,48.271833,HIXM
3,A2350200,A235020001,2007-04-01,754.0,7.459819,48.271833,HIXM
4,A2350200,A235020001,2007-05-01,783.0,7.459819,48.271833,HIXM
...,...,...,...,...,...,...,...
95624,A2350200,A235020001,2026-05-27,206.0,7.459819,48.271833,QmnJ
95625,A2350200,A235020001,2026-05-28,167.0,7.459819,48.271833,QmnJ
95626,A2350200,A235020001,2026-05-29,144.0,7.459819,48.271833,QmnJ
95627,A2350200,A235020001,2026-05-30,124.0,7.459819,48.271833,QmnJ


# 10. Save the dataframe

In [150]:
display(df.describe())

,date_obs_elab,resultat_obs_elab,longitude,latitude
count,27564,27564.000000,2.756400e+04,27564.000000
mean,2016-12-06 03:26:40.174140,2285.196924,7.459819e+00,48.271833
min,2007-01-01 00:00:00,-83.000000,7.459819e+00,48.271833
25%,2012-04-17 00:00:00,498.000000,7.459819e+00,48.271833
50%,2016-11-28 12:00:00,852.000000,7.459819e+00,48.271833
75%,2021-11-03 00:00:00,2622.000000,7.459819e+00,48.271833
max,2026-06-01 00:00:00,57193.000000,7.459819e+00,48.271833
std,NaN,3772.968456,2.664584e-15,0.000000


In [151]:
df_dates = df["date_obs_elab"]
max_date = datetime.datetime.strftime(df_dates.max(), "%Y%m%d")
min_date = datetime.datetime.strftime(df_dates.min(), "%Y%m%d")
print("min date:", min_date, "/", "max date:", max_date)

min date: 20070101 / max date: 20260601


In [152]:
base_name = f"hubeau_{HUBEAU_DATA_ENDPOINT}_{SITE}_{min_date}_{max_date}_1"
local_data_dirpath = os.path.abspath(os.path.join("..", "data", "hubeau"))
filepath = save_dataframe_as_csv(df, base_name=base_name, output_dirpath=local_data_dirpath)

In [153]:

filesize_in_kb = os.path.getsize(filepath) / 1024
print(f">>> saved : \"{filepath}\" ({len(df)} records, {filesize_in_kb:.0f} Kb)")

>>> saved : "c:\Users\nicolas\Repositories\Jedha\Shares\M11\nowcast-flood-risk\data\hubeau\hubeau_obs_elab_A2350200_20070101_20260601_1.csv" (27564 records, 1834 Kb)
